In [1]:
import pandas as pd

fn = 'olist-vrp-benchmark/best_known_solutions/best_known_costs_2024.csv'
df = pd.read_csv(fn)
df.head()

,Unnamed: 0,dataset,n_nodes,problem_id,vehicles,cost
0,0,SP,50,0,5,52972.797
1,1,SP,50,1,6,87505.200
2,2,SP,50,2,6,60889.000
3,3,SP,50,3,4,46312.895
4,4,SP,50,4,5,49052.300


In [2]:
df.groupby(['dataset', 'n_nodes'])['cost'].mean()

dataset  n_nodes
Rio      50          46768.249617
         100         78713.555422
         200        136114.955164
         300        189044.770664
         400        242397.485000
         500        296896.901562
SP       50          54426.322496
         100         89927.238230
         200        154254.829004
         300        218294.562852
         400        274352.659687
         500        329973.454648
Name: cost, dtype: float64

In [1]:
import torch
from tensordict.tensordict import TensorDict
from rl4co.envs.routing.cvrp.env import CVRPEnv 
from envs.symcvrp.env import SymCVRPEnv
from models.zoo.resym.model import ReSym
from models.nn.env_embeddings.symcvrp import SymVRPContext, SymVRPInitEmbedding
from rl4co.models.nn.env_embeddings.dynamic import StaticEmbedding
from models.zoo.resym.transforms import CompositeAugmentation
from rl4co.utils.ops import gather_by_index, unbatchify
import os 
from tqdm import tqdm
import pickle
from torch.utils.data import DataLoader

torch.manual_seed(1234)

def get_reward_unnormalized(data, actions):
    from rl4co.utils.ops import gather_by_index
    actions = torch.cat([torch.zeros_like(actions[..., :1]), actions], dim=-1)
    actions_next = torch.roll(actions, -1, dims=-1)
    batch_idx = torch.arange(actions.shape[0], device=actions.device)[:, None, None].expand_as(actions[..., :1])
    d = torch.tensor(data['distance_matrix'], device=actions.device)[batch_idx, actions, actions_next]

    return -d.sum(dim=-1).min(dim=-1)[0].mean()

# Utils function: we will normalize the coordinates of the VRP instances
def normalize_coord(coord:torch.Tensor) -> torch.Tensor:
    x, y = coord[:,:, 0], coord[:,:, 1]
    x_min, x_max = x.min(dim=-1, keepdim=True)[0], x.max(dim=-1, keepdim=True)[0]
    y_min, y_max = y.min(dim=-1, keepdim=True)[0], y.max(dim=-1, keepdim=True)[0]
    scaler = torch.maximum(x_max-x_min, y_max-y_min)
    x_scaled = (x - x_min) / scaler
    y_scaled = (y - y_min) / scaler
    coord_scaled = torch.stack([x_scaled, y_scaled], dim=-1)
    return coord_scaled

def olist_to_td(data, normalize=True):
    coords = torch.tensor(data['positions']).float()
    coords_norm = normalize_coord(coords) if normalize else coords
    demand = torch.tensor(data['demands'][:, 1:]).float()
    capacity = torch.tensor(data['capacities'][:, None]).float()
    n = coords.shape[1]
    td = TensorDict({
        'depot': coords_norm[:, 0,:],
        'locs': coords_norm[:, 1:,:],
        'demand': demand / capacity, # normalized demand
        'capacity': capacity, # original capacity, not needed for inference
    }, batch_size=coords_norm.shape[:1])
    return td

path = 'olist-vrp-benchmark/problem_instances'
files = sorted([fn for fn in os.listdir(path) if fn.endswith('.pkl') and 'test' in fn])
for fn in files:
    with open(os.path.join(path, fn), 'rb') as f:
        data = pickle.load(f)
    # td_orginal = olist_to_td(data, normalize=False)
    td = olist_to_td(data)

    num_loc = td['locs'].shape[1] + 1
    capacity = data['capacities'][0]
    env = CVRPEnv(generator_params={"num_loc": num_loc - 1, "capacity": capacity}, check_solution=False)
    device = "cuda" if torch.cuda.is_available() else "cpu"

    td_am = env.reset(td=td.clone()).to(device)
    td_vam = env.reset(td=td.clone()).to(device)

    model_am = ReSym(env=env, policy_kwargs={"encoder": "am", "decoder": "am", "pointer": None, "tanh_clipping": 50.0, })
    model_vam = ReSym(env=env, policy_kwargs={"encoder": "am", "decoder": "vam", "pointer": None, "tanh_clipping": 50.0, "logit_bias": True,})

    path_am = "outputs/checkpoints/cvrp100/am-am-None-100[]-multistart_sampling-0-False-0.0-v1/last.ckpt"
    path_vam = "outputs/checkpoints/cvrp100/am-vam-None-100[]-multistart_sampling-0-True-0.0-v1/last.ckpt"

    model_am.load_state_dict(torch.load(path_am, weights_only=False)['state_dict'])
    model_vam.load_state_dict(torch.load(path_vam, weights_only=False)['state_dict'])

    batch_size = min(256, (1000 // num_loc) ** 2)

    policy_am = model_am.policy.to(device)
    dataloader = DataLoader(td_am, batch_size=batch_size, collate_fn=lambda x: x)
    reward = []
    actions = []
    max_action_len = 0
    for batch in tqdm(dataloader):
        out_am = policy_am.forward(
            batch, 
            env, 
            phase="test", 
            decode_type="multistart_greedy", 
            num_starts=num_loc-1, 
            multistart=True,
        )
        reward.append(unbatchify(out_am['reward'], num_loc-1))
        actions.append(unbatchify(out_am['actions'], num_loc-1))
        max_action_len = max(max_action_len, out_am['actions'].shape[-1])
        del out_am
    reward = torch.cat(reward, dim=0)
    padded_actions = []
    for action in actions:
        if action.shape[-1] < max_action_len:
            padding = torch.zeros((*action.shape[:-1], max_action_len - action.shape[-1]), device=action.device, dtype=action.dtype)
            action = torch.cat([action, padding], dim=-1)
        padded_actions.append(action)
    actions = torch.cat(padded_actions, dim=0)
    # max_reward, max_idxs = reward.max(dim=-1)
    # action = gather_by_index(actions, max_idxs)
    # cost_am = -get_reward_unnormalized(data, action)
    cost_am = -get_reward_unnormalized(data, actions)

    policy_vam = model_vam.policy.to(device)
    dataloader = DataLoader(td_vam, batch_size=batch_size, collate_fn=lambda x: x)
    reward = []
    actions = []
    max_action_len = 0
    for batch in tqdm(dataloader):
        out_vam = policy_vam.forward(
            batch, 
            env, 
            phase="test", 
            decode_type="multistart_greedy", 
            num_starts=num_loc-1, 
            multistart=True,
        )
        reward.append(unbatchify(out_vam['reward'], num_loc-1))
        actions.append(unbatchify(out_vam['actions'], num_loc-1))
        max_action_len = max(max_action_len, out_vam['actions'].shape[-1])
        del out_vam
    reward = torch.cat(reward, dim=0)
    padded_actions = []
    for action in actions:
        if action.shape[-1] < max_action_len:
            padding = torch.zeros((*action.shape[:-1], max_action_len - action.shape[-1]), device=action.device, dtype=action.dtype)
            action = torch.cat([action, padding], dim=-1)
        padded_actions.append(action)
    actions = torch.cat(padded_actions, dim=0)
    # max_reward, max_idxs = reward.max(dim=-1)
    # action = gather_by_index(actions, max_idxs)
    # cost_vam = -get_reward_unnormalized(data, action)
    cost_vam = -get_reward_unnormalized(data, actions)
    
    print(f"{fn}: AM cost: {cost_am.item():.2f}, VAM cost: {cost_vam.item():.2f}")

/home/cpz/miniconda3/envs/rl4co/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/cpz/miniconda3/envs/rl4co/lib/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:210: Attribute 'env' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['env'])`.
/home/cpz/miniconda3/envs/rl4co/lib/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:210: Attribute 'policy' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['policy'])`.
100%|██████████| 3/3 [00:00<00:00,  5.36it/s]


vrp-test-size-100-rio-n_problems-256.pkl: AM cost: 95857.53, VAM cost: 89943.77


100%|██████████| 3/3 [00:00<00:00,  6.47it/s]


vrp-test-size-100-sp-n_problems-256.pkl: AM cost: 108957.08, VAM cost: 101230.15


100%|██████████| 1/1 [00:00<00:00, 20.12it/s]


vrp-test-size-20-rio-n_problems-256.pkl: AM cost: 29984.56, VAM cost: 29092.41


... (22 outputs left out)

100%|██████████| 64/64 [00:48<00:00,  1.33it/s]


vrp-test-size-500-rio-n_problems-256.pkl: AM cost: 470924.56, VAM cost: 362644.44


100%|██████████| 64/64 [00:47<00:00,  1.35it/s]

vrp-test-size-500-sp-n_problems-256.pkl: AM cost: 465172.38, VAM cost: 397954.22
